# RFM-UNet road extraction on Kaggle — DeepGlobe, tracked with MLflow

[RFM-UNet](https://github.com/FF7CA/RFMUNet) (Hybrid Frequency-Mamba UNet) with a VMamba-tiny
encoder, MAFM skip fusion and a frequency/spatial dual decoder. The model maths is upstream's;
this repo adapts it to a Kaggle session and logs to MLflow instead of a CSV file.

**Before you run anything**

1. *Settings → Accelerator* → **GPU T4 x2** (or P100). Also switch **Internet on** — the
   dependency install and the CUDA kernel build both need it.
2. *Add Input* → the **DeepGlobe Road Extraction Dataset**
   (`balraj98/deepglobe-road-extraction-dataset`).
3. Optional: *Add-ons → Secrets* → `MLFLOW_TRACKING_URI` (plus
   `MLFLOW_TRACKING_USERNAME` / `MLFLOW_TRACKING_PASSWORD`) to log to a remote tracking
   server. With no secret, everything goes to a local file store under
   `/kaggle/working/mlruns`, which is saved as notebook output.

**Read step 3 before starting a long run.** RFM-UNet is only practical with a compiled
selective-scan CUDA kernel, and upstream targets Ampere (compute ≥ 8.0) while Kaggle hands
out Turing T4s (7.5). Step 3 builds a kernel for the GPU you actually got and verifies it.

## 1. What hardware did we get?

In [ ]:
import os, subprocess, sys, textwrap

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
print(f"torch          {torch.__version__}")
print(f"cuda runtime   {torch.version.cuda}")
print(f"gpus visible   {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    major, minor = torch.cuda.get_device_capability(i)
    props = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {props.name}  compute {major}.{minor}  {props.total_memory / 1024**3:.1f} GB")

if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability(0)
    if major < 8:
        print(textwrap.dedent("""
        NOTE: upstream RFM-UNet asks for compute capability >= 8.0 and this GPU is older.
        Nothing in the model requires Ampere; the constraint comes from the prebuilt
        mamba-ssm wheels. Step 3 compiles a kernel for this exact architecture instead.
        """))
else:
    print("No GPU. Set Settings -> Accelerator -> GPU before training.")

## 2. Code and dependencies

The repo can arrive three ways: you are already running inside it, it is attached as a
Kaggle Dataset, or it gets cloned. All three end with `REPO` as the working directory,
and the cell refuses to continue unless the entry points are really there.

The clone is pinned to the **`rfmunet-kaggle`** branch — `main` still holds the old
U-Net notebook. Override with the `REPO_BRANCH` environment variable.

In [ ]:
from pathlib import Path
import shutil, subprocess, os

REPO_URL = "https://github.com/KorewaLidesu/unet-road-segmentation.git"
# The pipeline lives on this branch; main still holds the old U-Net notebook.
REPO_BRANCH = os.environ.get("REPO_BRANCH", "rfmunet-kaggle")
TARGET = Path("/kaggle/working/RFMUNet")

# A directory only counts as the repo if the entry points are actually in it.
# Without this check a clone of the wrong branch lands here and the failure
# surfaces two cells later as a missing requirements file.
SENTINELS = ("train_supervision.py", "requirements-kaggle.txt", "tools/prepare_deepglobe.py")


def is_repo(root):
    return all((root / name).exists() for name in SENTINELS)


def missing(root):
    return [name for name in SENTINELS if not (root / name).exists()]


def attached_copy():
    # A repo attached as a Kaggle Dataset, searched one and two levels deep.
    inputs = Path("/kaggle/input")
    if not inputs.exists():
        return None
    for depth in ("*/train_supervision.py", "*/*/train_supervision.py"):
        for hit in sorted(inputs.glob(depth)):
            if is_repo(hit.parent):
                return hit.parent
    return None


def find_repo():
    if is_repo(Path.cwd()):
        return Path.cwd()
    source = attached_copy()
    if source is not None:
        # /kaggle/input is read-only and training writes beside the code.
        if TARGET.exists():
            shutil.rmtree(TARGET)
        shutil.copytree(source, TARGET)
        print(f"copied attached repo from {source}")
        return TARGET
    if TARGET.exists() and not is_repo(TARGET):
        # Left over from an earlier run that cloned the wrong branch.
        print(f"discarding incomplete {TARGET} (missing {missing(TARGET)})")
        shutil.rmtree(TARGET)
    if not TARGET.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
                        REPO_URL, str(TARGET)], check=True)
    return TARGET


REPO = find_repo()
os.chdir(REPO)
if not is_repo(REPO):
    raise SystemExit(
        f"{REPO} is missing {missing(REPO)}. Set REPO_BRANCH if the pipeline "
        f"has moved off '{REPO_BRANCH}'."
    )
print("working in", REPO, f"(branch {REPO_BRANCH})")
print(sorted(p.name for p in REPO.iterdir() if not p.name.startswith(".")))

In [ ]:
!pip install -q -r requirements-kaggle.txt

## 3. The selective-scan CUDA kernel

This is the step that decides whether the run is viable.

The VMamba encoder's SSM blocks need a fused selective-scan kernel. Without one, the
vendored code falls back to a pure-PyTorch scan that loops over **every timestep in
Python** — 16384 iterations per block in the first stage at 512×512. It is numerically
correct and fine for a shape check, but roughly two orders of magnitude too slow to train,
and it materialises far more activation memory than a T4 has.

Two kernels work, and `geoseg/models/ADAMamba.py` will use whichever it finds:

| module | from | notes |
|---|---|---|
| `selective_scan_cuda` | `pip install mamba-ssm` | prebuilt wheels exist but target Ampere; a source build needs `nvcc` and is slow |
| `selective_scan_cuda_oflex` | VMamba's `kernels/selective_scan` | compiles for whatever `TORCH_CUDA_ARCH_LIST` you pass, so it is the reliable choice on a T4 |

Try the quick route first, then build if it did not take. **The build takes roughly
10–25 minutes.** Do it once, then save the wheel (last cell of this section) as a Kaggle
Dataset so later sessions skip it.

In [ ]:
# Quick route: an already-built wheel for this arch, if you saved one earlier.
# Attach that dataset and point WHEEL_DIR at it.
WHEEL_DIR = next((p for p in Path("/kaggle/input").glob("*/selective_scan*")), None) if Path("/kaggle/input").exists() else None
if WHEEL_DIR:
    print("found prebuilt wheels in", WHEEL_DIR)
    !pip install -q --no-index --find-links {WHEEL_DIR} selective_scan
else:
    print("no prebuilt wheel attached; trying mamba-ssm, then falling back to a source build")
    !pip install -q mamba-ssm==2.2.4 2>&1 | tail -5

In [ ]:
# Does a kernel exist, and is it actually usable on this GPU? An import can succeed
# while the cubin was built for another architecture, so call it once for real.
import importlib, sys, torch

MODULE = "geoseg.models.ADAMamba"


def probe():
    for name in ("selective_scan_cuda", "selective_scan_cuda_oflex"):
        try:
            importlib.import_module(name)
        except Exception as exc:
            print(f"  {name:26} not importable ({type(exc).__name__})")
            continue
        print(f"  {name:26} imported")
    # Reload only if it is already loaded: a reload re-runs timm's @register_model
    # decorators and prints an "Overwriting ... in registry" warning for each one.
    if MODULE in sys.modules:
        return importlib.reload(sys.modules[MODULE])
    return importlib.import_module(MODULE)


def verify_kernel(A):
    """Compare the kernel against the reference implementation on a small problem.

    Returns True only if a real kernel resolved AND agrees with the pure-PyTorch
    scan. An import proves nothing about whether the cubin matches this GPU.
    """
    backend = A.selective_scan_backend()
    print("resolved backend:", backend)
    if backend == "torch" or not torch.cuda.is_available():
        print("NO KERNEL -- run the build cell below before training for real.")
        return False

    torch.manual_seed(0)
    B, K, C, L, N = 1, 4, 8, 64, 16
    dev = "cuda"
    u = torch.randn(B, K * C, L, device=dev)
    delta = torch.rand(B, K * C, L, device=dev)
    Amat = -torch.rand(K * C, N, device=dev).exp()
    Bm = torch.randn(B, K, N, L, device=dev)
    Cm = torch.randn(B, K, N, L, device=dev)
    D = torch.randn(K * C, device=dev)
    bias = torch.randn(K * C, device=dev)
    try:
        fast = A.selective_scan_fn(u, delta, Amat, Bm, Cm, D, bias, True, backend=backend)
        ref = A.selective_scan_fn(u, delta, Amat, Bm, Cm, D, bias, True, backend="torch")
        err = (fast - ref).abs().max().item()
        print(f"kernel vs reference max abs error: {err:.3e}")
        assert err < 1e-2, "kernel disagrees with the reference implementation"
    except Exception as exc:
        print(f"KERNEL FAILED AT RUNTIME: {exc}")
        print("Run the build cell below.")
        return False

    print("KERNEL OK -- training is viable")
    return True


verify_kernel(probe())

In [ ]:
# Source build of VMamba's kernel for this GPU's exact architecture.
# Skip if the probe above already said KERNEL OK.
import torch, os
major, minor = torch.cuda.get_device_capability(0)
arch = f"{major}.{minor}"
print(f"building for compute {arch} (this takes 10-25 min)")

os.environ["TORCH_CUDA_ARCH_LIST"] = arch
os.environ["MAX_JOBS"] = str(min(4, os.cpu_count() or 2))

!test -d /kaggle/working/VMamba || git clone --depth 1 https://github.com/MzeroMiko/VMamba.git /kaggle/working/VMamba
!cd /kaggle/working/VMamba/kernels/selective_scan && TORCH_CUDA_ARCH_LIST={arch} MAX_JOBS={os.environ["MAX_JOBS"]} pip install -v . 2>&1 | tail -20

In [ ]:
# Re-probe after the build and run the same check, then stash the wheel so the
# next session can skip all this.
if not verify_kernel(probe()):
    raise SystemExit(
        "Still no working kernel after the build -- read the build output above. "
        "Training now would silently use the Python-loop fallback."
    )

!mkdir -p /kaggle/working/selective_scan_wheels
!cd /kaggle/working/VMamba/kernels/selective_scan && pip wheel . -w /kaggle/working/selective_scan_wheels 2>&1 | tail -3
!ls -la /kaggle/working/selective_scan_wheels 2>/dev/null
print("Save /kaggle/working/selective_scan_wheels as a Kaggle Dataset and attach it next time.")

## 4. Prepare the DeepGlobe split

Only DeepGlobe's `train/` folder is labelled — its `valid/` and `test/` folders are the
unlabelled competition holdouts — so the 6226 labelled pairs get split 80/10/10 here. The
split is on source image id *before* tiling, so crops of one scene never straddle splits.

Each 1024×1024 image becomes four 512×512 tiles (`--mode split`), which is what the model
trains at. Masks are re-encoded from 0/255 to class indices with **road = 0**, matching
`dpgb.PALETTE` and `tools/metric.py`. Feeding raw 0/255 masks in would make every road
pixel read as `ignore_index=255` and train on background alone — the prep script verifies
this rather than trusting it.

Add `--mode resize --fraction 0.25` for a fast first pass.

In [ ]:
import os
from pathlib import Path

# Where Kaggle mounts an attached dataset depends on how it was attached, so find
# it instead of hard-coding: DeepGlobe is whatever directory holds a *_sat.jpg with
# a matching *_mask.png. tools/prepare_deepglobe.py rglobs from here, so pointing
# at train/ is both precise and the fastest scan -- valid/ and test/ are the
# unlabelled competition holdouts and have no masks to find anyway.
SEARCH_ROOTS = (Path("/kaggle/input"), Path.home() / ".cache" / "kagglehub")


def find_deepglobe():
    for base in SEARCH_ROOTS:
        if not base.exists():
            continue
        for sat in base.rglob("*_sat.jpg"):
            if sat.with_name(sat.name.replace("_sat.jpg", "_mask.png")).exists():
                return sat.parent
    return None


found = find_deepglobe()
if found is None:
    present = sorted(p.name for p in Path("/kaggle/input").glob("*")) \
        if Path("/kaggle/input").exists() else []
    raise SystemExit(
        "No *_sat.jpg with a matching *_mask.png under " +
        " or ".join(str(r) for r in SEARCH_ROOTS) + ".\n"
        "Attach balraj98/deepglobe-road-extraction-dataset via Add Input -> Datasets, "
        "then rerun this cell.\n"
        f"Currently under /kaggle/input: {present}"
    )

os.environ["RAW_DEEPGLOBE_ROOT"] = str(found)
os.environ["DEEPGLOBE_ROOT"] = "/kaggle/working/data/deepglobe"

print("RAW_DEEPGLOBE_ROOT =", found)
print("labelled pairs here:", len(list(found.glob("*_sat.jpg"))))
print("dataset root holds:", sorted(p.name for p in found.parent.iterdir())[:10])

In [ ]:
!python tools/prepare_deepglobe.py     --src {os.environ["RAW_DEEPGLOBE_ROOT"]}     --dst {os.environ["DEEPGLOBE_ROOT"]}     --mode split --workers 4

In [ ]:
# Eyeball a few prepared pairs and confirm the label encoding.
import cv2, numpy as np, json
import matplotlib.pyplot as plt
from pathlib import Path

root = Path(os.environ["DEEPGLOBE_ROOT"])
print(json.dumps(json.loads((root / ".prepared.json").read_text()), indent=2))

masks = sorted((root / "train_masks").glob("*.png"))[:4]
fig, axes = plt.subplots(2, len(masks), figsize=(4 * len(masks), 8))
for col, mask_path in enumerate(masks):
    img = cv2.cvtColor(cv2.imread(str(root / "train_images" / f"{mask_path.stem}.jpg")), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
    axes[0, col].imshow(img); axes[0, col].set_title(mask_path.stem, fontsize=8)
    axes[1, col].imshow(mask == 0, cmap="gray")
    axes[1, col].set_title(f"road px {np.mean(mask == 0):.3f} | values {np.unique(mask).tolist()}", fontsize=8)
    for row in (0, 1):
        axes[row, col].axis("off")
plt.tight_layout(); plt.show()

## 5. MLflow

No configuration needed: without a tracking URI everything lands in
`/kaggle/working/mlruns`, which survives as notebook output. Download it and read it with
`mlflow ui --backend-store-uri ./mlruns`.

If a `MLFLOW_TRACKING_URI` Kaggle Secret exists it is used instead, and credentials come
from the same place. The run id is cached next to the checkpoints, so if the session is
killed and you rerun, training resumes into the *same* MLflow run rather than starting a
second one.

In [ ]:
from tools import mlflow_utils

found = mlflow_utils.load_kaggle_secrets()
print("secrets picked up:", found or "none (using the local file store)")

os.environ["OUT_ROOT"] = "/kaggle/working"
print("tracking uri:", mlflow_utils.resolve_tracking_uri("/kaggle/working/mlruns"))

## 6. Smoke test

Two training batches and two validation batches at the real resolution. This catches
config mistakes, shape errors, an OOM at this batch size, and a broken MLflow target in
about a minute — rather than at hour three of a real run.

In [ ]:
!rm -rf /kaggle/working/smoke
!OUT_ROOT=/kaggle/working/smoke  MAX_EPOCH=1 LIMIT_TRAIN_BATCHES=2 LIMIT_VAL_BATCHES=2  NUM_SANITY_VAL_STEPS=0 ACCUM_GRAD_BATCHES=1 NUM_WORKERS=2  MLFLOW_EXPERIMENT_NAME=RFMUNet-smoke MLFLOW_RUN_NAME=smoke  MLFLOW_DEFAULT_DIR=/kaggle/working/smoke/mlruns MLFLOW_SYSTEM_METRICS=0  LOG_PREDICTION_IMAGES=1 PREDICTION_IMAGE_EVERY_N_EPOCHS=1 MAX_TIME=  python train_supervision.py -c config/kaggle/RFMUNet.py

## 7. Train

Defaults: batch 2 × 6 accumulation steps (upstream's effective batch of 12), 16-bit mixed
precision, gradient checkpointing on the encoder, and a wall-clock stop at 8 hours so the
final checkpoint is written before Kaggle ends the session.

Training runs as a subprocess rather than in-process. That is deliberate: it makes real
DDP work (set `GPUS=2 STRATEGY=ddp` on a T4 x2 box), where Lightning's in-notebook
strategy is unreliable with dataloader workers.

Rerunning this cell resumes from `last.ckpt` and continues the same MLflow run, so a
105-epoch schedule can be spread over several sessions. Lower `MAX_EPOCH` for a first
real run — the cosine schedule restarts at 15, 45, 105, so 45 is a natural stopping point.

In [ ]:
!OUT_ROOT=/kaggle/working  DEEPGLOBE_ROOT=/kaggle/working/data/deepglobe  MAX_EPOCH=45  TRAIN_BATCH_SIZE=2 VAL_BATCH_SIZE=2 ACCUM_GRAD_BATCHES=6  PRECISION=16-mixed USE_GRAD_CHECKPOINT=1  GPUS=1 STRATEGY=auto NUM_WORKERS=2  MAX_TIME=00:08:00:00  MLFLOW_EXPERIMENT_NAME=RFMUNet-DeepGlobe  PREDICTION_IMAGE_EVERY_N_EPOCHS=5  python train_supervision.py -c config/kaggle/RFMUNet.py

## 8. Evaluate

Scores the held-out test split with horizontal/vertical flip TTA and writes the predicted
masks. clDice and APLS are computed here — they skeletonise and graph every image, which
is why they are off during training — and all the metrics are logged onto the training run
so they sit beside its curves.

In [ ]:
!OUT_ROOT=/kaggle/working DEEPGLOBE_ROOT=/kaggle/working/data/deepglobe  python road_seg_test.py     -c config/kaggle/RFMUNet.py     -o /kaggle/working/results/deepglobe/RFMUNet     --rgb -t lr --batch-size 2

## 9. Read the run back

In [ ]:
import mlflow, pandas as pd
import matplotlib.pyplot as plt

mlflow.set_tracking_uri(mlflow_utils.resolve_tracking_uri("/kaggle/working/mlruns"))
runs = mlflow.search_runs(experiment_names=["RFMUNet-DeepGlobe"], order_by=["start_time DESC"])
cols = [c for c in ["run_id", "status", "metrics.val_mIoU", "metrics.val_IoU_Road",
                    "metrics.val_F1_Road", "metrics.test_IoU_Road", "metrics.test_clDice",
                    "metrics.test_APLS", "params/env/selective_scan"] if c in runs.columns]
display(runs[cols].head())

In [ ]:
# Training curves for the newest run.
run_id = runs.iloc[0].run_id
client = mlflow.tracking.MlflowClient()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
panels = [
    ("loss", ["train_loss_epoch", "val_loss"]),
    ("road IoU", ["train_IoU_Road", "val_IoU_Road"]),
    ("mIoU", ["train_mIoU", "val_mIoU"]),
]
for ax, (title, keys) in zip(axes, panels):
    for key in keys:
        history = client.get_metric_history(run_id, key)
        if history:
            ax.plot([h.step for h in history], [h.value for h in history], label=key, marker=".")
    ax.set_title(title); ax.set_xlabel("step"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

print("prediction previews logged under val_predictions/ in the run's artifacts")

## 10. Keep the outputs

Kaggle keeps `/kaggle/working` (20 GB) as notebook output. Zip the tracking store and the
checkpoint so the next session — or your laptop — can pick them up. Save this notebook's
output as a Kaggle Dataset and attach it to continue training later.

In [ ]:
!du -sh /kaggle/working/mlruns /kaggle/working/model_weights 2>/dev/null
!cd /kaggle/working && zip -qr mlruns.zip mlruns && ls -lh mlruns.zip
!ls -lh /kaggle/working/model_weights/deepglobe/RFMUNet/
!rm -rf /kaggle/working/data /kaggle/working/VMamba   # big and reproducible; drop from the output

### Continuing in a later session

1. Save this session's output as a Kaggle Dataset.
2. In the new notebook, attach it and set `WEIGHTS_PATH` and `MLFLOW_DEFAULT_DIR` to copies
   of `model_weights/...` and `mlruns/` under `/kaggle/working` (both need to be writable).
3. Rerun step 7. `last.ckpt` is picked up automatically and the metrics continue in the
   same MLflow run.